In [1]:
import numpy as np
import pandas as pd
from prophet import Prophet
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
# 1. LOAD DATA & FORMAT FOR PROPHET
df_prophet = pd.read_csv("../data/processed/demand_features_v3.csv", parse_dates=["date"])

all_results_prophet = []
trained_models_prophet = {}
for sku in df_prophet["sku_id"].unique():
    sku_df = df_prophet[df_prophet["sku_id"] == sku].copy()
    
    sku_df = sku_df.rename(columns={"date": "ds", "units_sold": "y"})
    
    # 80/20 chronological split
    split_idx = int(len(sku_df) * 0.8)
    train_prophet = sku_df.iloc[:split_idx]
    test_prophet = sku_df.iloc[split_idx:]
    # 2. INSTANTIATE & TRAIN PROPHET
    model = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=True,
        daily_seasonality=False,
        seasonality_mode='multiplicative' # Multiplicative handles scaling trends better
    )
    
    # CRITICAL: Tell Prophet to pay attention to our external shocks
    model.add_regressor('is_promo')
    model.add_regressor('is_stockout')
    
    # Fit the model
    model.fit(train_prophet)
    # 3. PREDICT & EVALUATE
    # Create a future dataframe that perfectly matches the test set dates
    future = test_prophet[['ds', 'is_promo', 'is_stockout']]
    
    # Generate Forecast
    forecast = model.predict(future)
    
    # Extract predictions (Prophet stores predictions in the 'yhat' column)
    preds_actual = forecast['yhat'].values
    y_test_actual = test_prophet['y'].values
    
    # Ensure no negative predictions (Prophet can sometimes dip below 0)
    preds_actual = np.maximum(0, preds_actual)
    
    # Calculate Metrics
    rmse = np.sqrt(mean_squared_error(y_test_actual, preds_actual))
    mae = mean_absolute_error(y_test_actual, preds_actual)
    mape = np.mean(np.abs((y_test_actual - preds_actual) / y_test_actual)) * 100
    r2 = r2_score(y_test_actual, preds_actual)
    
    all_results_prophet.append({
        "model": "Prophet", 
        "sku_id": sku, 
        "RMSE": rmse, 
        "MAE": mae, 
        "MAPE": mape, 
        "R2": r2
    })
    
    trained_models_prophet[sku] = model
# 4. SUMMARIZE RESULTS
results_df_prophet = pd.DataFrame(all_results_prophet)
summary_df_prophet = results_df_prophet.groupby("model")[["RMSE", "MAE", "MAPE", "R2"]].mean().round(3)
print("\n--- Prophet Performance Summary ---")
print(summary_df_prophet)
# Save results
results_df_prophet.to_csv("../data/processed/prophet_results_v3.csv", index=False)
summary_df_prophet.to_csv("../data/processed/prophet_summary_v3.csv")

23:23:08 - cmdstanpy - INFO - Chain [1] start processing
23:23:10 - cmdstanpy - INFO - Chain [1] done processing
23:23:10 - cmdstanpy - INFO - Chain [1] start processing
23:23:10 - cmdstanpy - INFO - Chain [1] done processing
23:23:10 - cmdstanpy - INFO - Chain [1] start processing
23:23:10 - cmdstanpy - INFO - Chain [1] done processing
23:23:11 - cmdstanpy - INFO - Chain [1] start processing
23:23:11 - cmdstanpy - INFO - Chain [1] done processing



--- Prophet Performance Summary ---
           RMSE     MAE    MAPE     R2
model                                 
Prophet  56.725  33.116  13.606  0.515


In [1]:
import pandas as pd
import numpy as np
from prophet import Prophet
from sklearn.metrics import mean_squared_error, r2_score

df_raw = pd.read_csv("../data/processed/demand_features_v4_1M.csv", parse_dates=["date"])

# Aggregate store data up to the national level for Prophet
df_prophet = df_raw.groupby(['date', 'sku_id']).agg({
    'units_sold': 'sum',
    'is_promo': 'max',      # Flag if a promo occurred anywhere
    'is_stockout': 'max'    # Flag if a stockout occurred anywhere
}).reset_index()

all_results_prophet = []
for sku in df_prophet["sku_id"].unique():
    sku_df = df_prophet[df_prophet["sku_id"] == sku].copy()
    sku_df = sku_df.rename(columns={"date": "ds", "units_sold": "y"})
    
    split_idx = int(len(sku_df) * 0.8)
    train_prophet = sku_df.iloc[:split_idx]
    test_prophet = sku_df.iloc[split_idx:]
    
    model = Prophet(yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False, seasonality_mode='multiplicative')
    model.add_regressor('is_promo')
    model.add_regressor('is_stockout')
    
    model.fit(train_prophet)
    
    future = test_prophet[['ds', 'is_promo', 'is_stockout']]
    forecast = model.predict(future)
    
    preds_actual = np.maximum(0, forecast['yhat'].values)
    y_test_actual = test_prophet['y'].values
    
    rmse = np.sqrt(mean_squared_error(y_test_actual, preds_actual))
    r2 = r2_score(y_test_actual, preds_actual)
    
    all_results_prophet.append({"model": "Prophet_Global", "sku_id": sku, "RMSE": rmse, "R2": r2})

print(pd.DataFrame(all_results_prophet).groupby("model")[["RMSE", "R2"]].mean().round(3))

19:55:49 - cmdstanpy - INFO - Chain [1] start processing
19:55:50 - cmdstanpy - INFO - Chain [1] done processing
19:55:50 - cmdstanpy - INFO - Chain [1] start processing
19:55:50 - cmdstanpy - INFO - Chain [1] done processing
19:55:50 - cmdstanpy - INFO - Chain [1] start processing
19:55:51 - cmdstanpy - INFO - Chain [1] done processing
19:55:51 - cmdstanpy - INFO - Chain [1] start processing
19:55:51 - cmdstanpy - INFO - Chain [1] done processing


                   RMSE     R2
model                         
Prophet_Global  217.608  0.789
